# Layout choice, strong parameterizations and physical coordinates
[Proof](../13_acquisition_only_budget_choice.md). Keep the original round-trip, expected-risk and full-side-input controls.

In [ ]:
import numpy as np
from experiments.sampled_conditioning.core import sampled_design,expected_plugin_kl,retained_matrix,choose_pattern,encode_header,decode_header
from experiments.sampled_conditioning.run import evaluate_design
from experiments.sampled_conditioning.parameterization_controls import representations,blocks,kl,lowrank_covariance
t,A=sampled_design(1024,.2,30.)
J=A.T@A/.25
costs=[expected_plugin_kl(J,retained_matrix(J,k)) for k in range(3)]
selected=choose_pattern(J)
assert costs[selected]==min(costs)
header=encode_header(np.arange(4.),J,selected)
score,decoded=decode_header(header)
assert len(header)==11
np.testing.assert_allclose(decoded,retained_matrix(J,selected))
print('expected KL, layout',costs,selected)

In [ ]:
rng=np.random.default_rng(44)
beta=rng.normal(size=(40000,4));noise=rng.normal(size=(40000,len(A)))
result=evaluate_design(A,beta,noise,'within')
se=result['kl'].std(ddof=1)/np.sqrt(len(beta))
assert abs(result['kl'].mean()-result['expected_kl'])<5*se
for arm in ['diag','within','magnitude','risk_oracle','full']:
    np.testing.assert_allclose(evaluate_design(A,beta[:64],noise[:64],arm,True)['kl'],0.,atol=1e-10)
plugin=evaluate_design(A,beta[:256],noise[:256],'within')
moment=evaluate_design(A,beta[:256],noise[:256],'moment_within')
assert np.all(moment['kl']<=plugin['kl']+1e-9)
np.testing.assert_allclose(moment['kl'],moment['expected_kl'],atol=1e-10)
print('prior-predictive MC, exact',result['kl'].mean(),result['expected_kl'])

## Fixed 19-statistic counterexample, distinct from the adaptive 11-scalar study
Three modes, protocol-shared 4+2 grouping. No per-event partition is chosen.

In [ ]:
j=.2*np.eye(6)+2*np.ones((6,6));h=np.array([1.,-.3,.4,.7,-.8,.2]);groups=((0,1,2,3),(4,5))
arms=representations(j,h,groups);mu,S,_=arms['full']
for name,(mean,cov,budget) in arms.items():
    print(name,budget,kl(mu,S,mean,cov))
mn,sn,_=arms['natural_blocks'];mm,sm,_=arms['moment_blocks']
marginal_sum=sum(kl(mu[list(g)],S[np.ix_(g,g)],mn[list(g)],sn[np.ix_(g,g)]) for g in groups)
np.testing.assert_allclose(kl(mu,S,mn,sn)-kl(mu,S,mm,sm),marginal_sum,atol=1e-12)
for g in groups:
    assert np.linalg.eigvalsh((S-sn)[np.ix_(g,g)]).min()>-1e-12
np.testing.assert_allclose(lowrank_covariance(np.eye(6),S,6),S,atol=1e-12)

## Phase change includes the prior; a cheaper covariant control exists
Only groups formed from full cosine/sine pairs qualify. Arbitrary coordinate matchings do not.

In [ ]:
rng=np.random.default_rng(13);x=rng.normal(size=(6,6));j=x@x.T+np.eye(6)
q0=np.diag([1.,2.,3.,4.,5.,6.]);h0=rng.normal(size=6);b=rng.normal(size=6)
u=np.zeros((6,6))
for k,angle in enumerate([.2,.9,-.7]):
    c,s=np.cos(angle),np.sin(angle);u[2*k:2*k+2,2*k:2*k+2]=[[c,-s],[s,c]]
np.testing.assert_allclose(blocks(u@j@u.T,groups),u@blocks(j,groups)@u.T,atol=1e-12)
q=q0+blocks(j,groups);qp=u@q0@u.T+blocks(u@j@u.T,groups)
np.testing.assert_allclose(np.linalg.solve(qp,u@(h0+b)),u@np.linalg.solve(q,h0+b),atol=1e-12)
iso=lambda a: np.diag(np.repeat([np.trace(a[k:k+2,k:k+2])/2 for k in range(0,6,2)],2))
np.testing.assert_allclose(iso(u@j@u.T),u@iso(j)@u.T,atol=1e-12)
assert np.linalg.norm(np.diag(np.diag(u@j@u.T))-u@np.diag(np.diag(j))@u.T)>.1
print('phase controls passed; covariance alone does not justify full blocks')

Moment blocks are the fixed-partition forward-KL reference, but require full encoder inference. A goal-only representation may be cheaper yet cannot represent every possible task. Cost and target tests—not this Notebook count—decide whether a learned condition is useful.

In [ ]:
print("THEORY_DEMO_PASS::13_acquisition_only_budget_choice")